# Kısmi Korelasyon (Partial Correlation)
Kısmi Korelasyon, iki değişken arasındaki korelasyonu hesaplarken, 
**üçüncü bir değişkenin etkisini istatistiksel olarak kontrol ederek/
sabitleyerek** "SAF" ilişkiyi ölçer. Bu, ANCOVA'da öğrendiğimiz 
"karıştırıcı değişkeni kontrol etme" mantığının, korelasyona uygulanmış 
hali.

## Neden Gerekli?
Az önceki Korelasyon Matrisi örneğinde, Reklam-Müşteri Sayısı arasında 
güçlü bir korelasyon (0.86) bulmuştuk. Ama biliyoruz ki **Satış** de 
hem Reklam'la (0.88) hem Müşteri Sayısı'yla (0.71) güçlü ilişkili. 
**Soru şu:** Reklam ile Müşteri Sayısı arasındaki o güçlü ilişki, 
gerçekten **doğrudan** mı, yoksa aslında **Satış üzerinden dolaylı** mı 
oluşuyor? (Tıpkı ANCOVA'daki Kanal-Yaş-Memnuniyet örneğimizdeki gibi — 
"gerçek sebep bu mu, yoksa gizli bir aracı mı var" sorusu.)

Kısmi korelasyon, "Satış'ı sabitlersek, Reklam ile Müşteri Sayısı 
arasında hâlâ bir ilişki kalıyor mu?" sorusuna cevap verir.

## Formül (Basitleştirilmiş, 3 Değişkenli Durum)
$$r_{XY \cdot Z} = \frac{r_{XY} - r_{XZ} \cdot r_{YZ}}{\sqrt{(1-r_{XZ}^2)(1-r_{YZ}^2)}}$$

Burada $r_{XY \cdot Z}$, "Z'nin etkisi kontrol edildiğinde X ile Y 
arasındaki korelasyon" demek.

## Python'da Kullanımı
```python
import pingouin as pg

pg.partial_corr(data=df, x='Reklam', y='Musteri_Sayisi', covar='Satis')
```

## Yorumlama
Eğer kısmi korelasyon, **ham (kontrolsüz) korelasyondan çok daha düşükse**, bu, ilk bulduğumuz ilişkinin büyük ölçüde **üçüncü değişken üzerinden dolaylı** olduğunu gösterir. ANCOVA'daki Kanal-Yaş örneğinde olduğu gibi. 
Eğer kısmi korelasyon **ham korelasyona yakın kalıyorsa**, ilişki **gerçekten doğrudan** demektir, üçüncü değişkenin bunda büyük payı yoktur.

In [1]:
import numpy as np
import pandas as pd
import pingouin as pg

np.random.seed(42)

reklam = np.random.normal(500, 100, 50)
satis = reklam * 0.8 + np.random.normal(0, 50, 50)
sikayet = np.random.normal(10, 3, 50)
musteri_sayisi = reklam * 0.3 + np.random.normal(0, 20, 50)

df = pd.DataFrame({
    'Reklam': reklam,
    'Satis': satis,
    'Sikayet': sikayet,
    'Musteri_Sayisi': musteri_sayisi
})

# --- Ham (kontrolsüz) korelasyon ---
ham_korelasyon = df[['Reklam', 'Musteri_Sayisi']].corr().iloc[0,1]
print(f"Ham Korelasyon (Reklam - Müşteri Sayısı): {ham_korelasyon:.4f}")

# --- Kısmi korelasyon (Satış kontrol edilerek) ---
kismi_sonuc = pg.partial_corr(data=df, x='Reklam', y='Musteri_Sayisi', covar='Satis')
print("\nKısmi Korelasyon (Satış kontrol edilerek):")
print(kismi_sonuc)

Ham Korelasyon (Reklam - Müşteri Sayısı): 0.8598

Kısmi Korelasyon (Satış kontrol edilerek):
          n         r          CI95         p_val
pearson  50  0.703029  [0.53, 0.82]  1.780100e-08


### Sonuç
Reklam ile Müşteri Sayısı arasındaki ham korelasyon 0.86 iken, Satış'ın 
etkisi kontrol edildiğinde bu ilişki 0.70'e düşmüştür (p<0.001). 
Korelasyonun düşmesi, ilk bulunan ilişkinin bir kısmının Satış 
üzerinden dolaylı olduğuna işaret etmektedir. Ancak kısmi korelasyonun 
hâlâ güçlü ve istatistiksel olarak anlamlı kalması, Reklam ile Müşteri 
Sayısı arasında Satış'tan bağımsız, doğrudan bir ilişkinin de var 
olduğunu göstermektedir.